In [1]:
#============
#CELL NO-: 01
#In this cell we will load the dataset and check the number of rows and columns.
#============

import pandas as pd

FILE = r"F:\c2-iot2\data\raw\train_test_network.csv"

df = pd.read_csv(FILE, na_values=['-'], low_memory=False)

print("Number of rows :", df.shape[0])
print("Number of columns :", df.shape[1])

Number of rows : 211043
Number of columns : 44


In [2]:
#============
#CELL NO-: 02
#In this section we will check what are the attack types and how many
#rows with each attack type, also how many rows with normal vs attack data
#============

print("Attack types")
print(df['type'].value_counts())

print()  

print("Binary label (0=normal and 1=attack)")
print(df['label'].value_counts())

Attack types
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: int64

Binary label (0=normal and 1=attack)
label
1    161043
0     50000
Name: count, dtype: int64


In [3]:
# ============
# CELL NO: 03
# Check data types of all 44 columns
# ============

df.dtypes

src_ip                        str
src_port                    int64
dst_ip                        str
dst_port                    int64
proto                         str
service                       str
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                    str
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_query                     str
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                        str
dns_RD                        str
dns_RA                        str
dns_rejected                  str
ssl_version                   str
ssl_cipher                    str
ssl_resumed                   str
ssl_established               str
ssl_subject                   str
ssl_issuer                    str
http_trans_dep

In [4]:
# ============
# CELL NO: 04
# Check duplicate rows across the raw dataset to prevent data leakage before splitting.
# ============

dup_count = df.duplicated().sum()

print("=" * 50)
print("Issue 1: Duplicate rows")
print("=" * 50)
print(f"Duplicate row count : {dup_count:,}")
print(f"Out of total rows   : {100 * dup_count / len(df):.2f}%")
print("\nWhich attack types:")
print(df[df.duplicated(keep=False)]["type"].value_counts())

Issue 1: Duplicate rows
Duplicate row count : 20,569
Out of total rows   : 9.75%

Which attack types:
type
normal        8123
xss           8039
ransomware    6774
backdoor      2512
dos           2016
password       209
injection       48
ddos            14
mitm             4
Name: count, dtype: int64


In [5]:
# ============
# CELL NO: 05
# Check if src_ip leaks the label (pure IPs that belong to only one class).
# ============

ip_table = pd.crosstab(df["src_ip"], df["label"])

pure_ips = (ip_table.min(axis=1) == 0).sum()
total_ips = len(ip_table)

print(f"Total unique src_ip count       : {total_ips}")
print(f"Belonging to only one label     : {pure_ips}")
print(f"Percentage                      : {100 * pure_ips / total_ips:.1f}%")

Total unique src_ip count       : 51
Belonging to only one label     : 43
Percentage                      : 84.3%


In [6]:
#============
#CELL NO-: 05b
#Here we just look at the top 10 src_ip addresses by row count
#to see which IPs dominate the dataset
#============

print("Top src_ip by row count:")
print(df['src_ip'].value_counts().head(10))

Top src_ip by row count:
src_ip
192.168.1.30     61633
192.168.1.31     30355
192.168.1.32     27227
192.168.1.193    25000
192.168.1.152    21893
192.168.1.33      9439
192.168.1.37      7500
192.168.1.195     4208
192.168.1.180     3708
192.168.1.190     3179
Name: count, dtype: int64


In [7]:
#============
#CELL NO-: 06
#In this section we check if dst_port also reveals the attack type
#Just like src_ip, if a port number is strongly linked to one attack type,
#a model could use the port as a shortcut instead of learning real behaviour
#We check specific well-known ports: 8080, 4444, 445, 21
#============

print("--- Does dst_port reveal the attack type? ---")
port_table = pd.crosstab(df['dst_port'], df['type'])

for port in [8080, 4444, 445, 21]:
    if port in port_table.index:
        row = port_table.loc[port]
        row = row[row > 0]          # only the non-zero ones
        print(f"\nPort {port} -> {row.to_dict()}")

--- Does dst_port reveal the attack type? ---

Port 8080 -> {'backdoor': 10235, 'ddos': 23, 'scanning': 71}

Port 4444 -> {'backdoor': 6, 'ransomware': 2364, 'scanning': 14}

Port 445 -> {'backdoor': 126, 'ransomware': 13235, 'scanning': 88}

Port 21 -> {'dos': 290, 'password': 5575, 'scanning': 15}


In [8]:
#============
#CELL NO-: 07
#In this section we check how much missing data (NaN) each column has
#We calculate the missing percentage for every column and sort so the
#emptiest columns come first
#Then we find columns that are more than 95% empty - these have very
#little useful information for the model
#============

n = len(df)

# Calculate the NaN percentage for each column
missing = (df.isna().sum() / n * 100).round(2)

# Sort so the emptiest ones come first
missing = missing.sort_values(ascending=False)

print("=" * 55)
print("NaN percentage for each column")
print("=" * 55)
print(missing.to_string())

NaN percentage for each column
ssl_issuer                99.99
http_orig_mime_types      99.99
ssl_subject               99.99
weird_addl                99.93
http_resp_mime_types      99.90
http_version              99.86
http_method               99.86
http_trans_depth          99.86
http_user_agent           99.86
http_uri                  99.86
weird_notice              99.83
weird_name                99.83
ssl_established           99.81
ssl_resumed               99.81
ssl_version               99.81
ssl_cipher                99.81
dns_query                 83.49
dns_rejected              83.41
dns_AA                    83.41
dns_RD                    83.41
dns_RA                    83.41
service                   62.56
src_ip                     0.00
dst_ip                     0.00
src_port                   0.00
duration                   0.00
proto                      0.00
dst_port                   0.00
src_pkts                   0.00
missed_bytes               0.00
conn_stat

In [9]:
#============
#CELL NO-: 07b
#Here we list out only the columns that are more than 95% empty
#These will be dropped in Notebook 02 (preprocessing) because they
#don't have enough real data to be useful
#============

# Ones that are more than 95% empty
mostly_empty = missing[missing > 95].index.tolist()

print("=" * 55)
print(f"Columns more than 95% empty : {len(mostly_empty)}")
print("=" * 55)
for c in mostly_empty:
    print(f"  {c:28s} {missing[c]:6.2f}%")

Columns more than 95% empty : 16
  ssl_issuer                    99.99%
  http_orig_mime_types          99.99%
  ssl_subject                   99.99%
  weird_addl                    99.93%
  http_resp_mime_types          99.90%
  http_version                  99.86%
  http_method                   99.86%
  http_trans_depth              99.86%
  http_user_agent               99.86%
  http_uri                      99.86%
  weird_notice                  99.83%
  weird_name                    99.83%
  ssl_established               99.81%
  ssl_resumed                   99.81%
  ssl_version                   99.81%
  ssl_cipher                    99.81%


## Summary — Findings for Notebook 02 (Preprocessing)

The issues discovered in this notebook will be fixed in Notebook 02 (preprocessing):

| # | Issue | Finding | Fix in Notebook 02 |
|---|---|---|---|
| 1 | Duplicate rows | 20,569 rows (9.75%) | `drop_duplicates()` — before any split |
| 2 | Columns >95% missing | 16 columns (ssl_*, http_* text fields, weird_*) | `drop(columns=...)` |
| 3 | `src_ip` leakage | 84.3% of IPs (43/51) are single-label pure | Hard-exclude from Set B |
| 4 | `dst_port` leakage | Ports strongly correlate with specific attack types | Hard-exclude from Set B |
| 5 | `dns_query` leakage | 83.49% missing, testbed IP embedded in query strings | Hard-exclude from Set B (manual, since it falls below the 95% threshold) |
| 6 | Class imbalance | mitm = 1,043 (severe minority) vs 20,000 for other classes | Use Macro-F1 as the primary metric |
| 7 | Boolean-as-string columns | 24 columns are str type, including dns_AA/RD/RA/rejected, ssl_resumed/established stored as "T"/"F" text | Map to 1/0 (with safe existence-check) |
| 8 | `service` missingness | 62.56% missing, a meaningful pattern (non-standard protocols) | Keep as NaN category instead of filling |

**Core novelty framing — important to remember:**

> The five identifiers (`src_ip`, `dst_ip`, `src_port`, `dst_port`, `dns_query`) are excluded not just because statistics showed leakage — but because they are fields an attacker can manipulate. This threat-model-driven exclusion is applied before any statistical feature selection.

**Decision note (2026-10-15):** This project uses the **original (imbalanced) TON_IoT dataset** (211,043 raw rows), not a SMOTE/SMOTENC-rebalanced version — chosen because Macro-F1 already handles class imbalance at the metric level, the core T2 robustness novelty does not depend on class balance, and the original class proportions (e.g. mitm at 0.49%) better reflect real-world attack rarity than a synthetically rebalanced distribution.

**Note:** In this notebook, the `df` variable remains completely unchanged from its original raw state. No drop/fix operations were performed here — all of them will be executed in Notebook 02 (preprocessing).

**Next step:** Notebook 02 — Data Preprocessing (deduplication, dropping columns, exclusion, boolean mapping, defining Set A/Set B).